In [1]:
import pandas as pd                                              # выполняем необъодимые импорты
import numpy as np
from clickhouse_connect import get_client
from special_modules.db_connector import connect_to_db
pd.set_option('display.max_rows', 200)                           # настройка отображения выводимых таблиц (чтобы не перегружать ноутбук)
pd.set_option('display.max_columns', 50)

In [2]:
client = connect_to_db()

In [3]:
sql_tickets_base = '''
WITH base_joined AS(
	SELECT 
		s.film_id AS film_id,
		f.price_ticket AS price_ticket,
		f.film_name AS film_name,
		fp.session_date AS session_date,
		fp.part_of_day AS part_of_day,
		fp.client_id AS client_id,
		fp.type_pay AS type_pay,
		fp.pay_at AS pay_at,
		fp.cinema_id AS cinema_id,
		fp.cinema_room_id AS cinema_room_id,
		dcr.count_seats AS count_seats,
		dcr.is_vip AS is_vip,
		dcr.is_3d AS is_3d,
		c.discount_perc AS discount_perc,
		(f.price_ticket*c.discount_perc)/100 AS discount_amount,
		f.price_ticket - (f.price_ticket*c.discount_perc)/100 AS ticket_price_final,
		toMonth(fp.pay_at) AS month,
		multiIf(
			dcr.is_vip = 1 AND dcr.is_3d = 1, 'VIP+3D', 
			dcr.is_vip = 0 AND dcr.is_3d = 1, '3D', 
			dcr.is_vip = 1 AND dcr.is_3d = 0, 'VIP',
			'Standart') cinema_room_type,
		concat(
			toString(fp.session_date),' ',
			toString(fp.part_of_day),' ',
			toString(fp.cinema_id),' ',
			toString(fp.cinema_room_id)) AS session_key --технический столбец
	FROM alpha_cinema.films_payments fp 
	LEFT JOIN alpha_cinema.schedule s 
		ON fp.session_date = s.session_date 
		AND fp.part_of_day = s.part_of_day 
		AND fp.cinema_id = s.cinema_id 
		AND fp.cinema_room_id = s.cinema_room_id
	LEFT JOIN alpha_cinema.films f 
		ON s.film_id = f.film_id 
	LEFT JOIN alpha_cinema.clients c 
		ON c.client_id = fp.client_id 
	LEFT JOIN alpha_cinema.dict_cinema_rooms dcr 
		ON fp.cinema_id = dcr.cinema_id
		AND fp.cinema_room_id = dcr.cinema_room_id),
base_enriched AS(
	SELECT
		*,
		ROW_NUMBER() OVER(PARTITION BY client_id ORDER BY pay_at) AS client_purchase_number,
		COUNT() OVER(PARTITION BY session_key) AS session_tickets_sold,
		SUM(ticket_price_final) OVER(PARTITION BY session_key) AS session_revenue
	FROM base_joined bj)
SELECT *, ROUND(session_tickets_sold/count_seats*100,2) AS occupancy_rate
FROM base_enriched
ORDER BY film_id
'''

In [10]:
df_tickets = client.query_df(sql_tickets_base)
df_tickets.head(2)

,film_id,price_ticket,film_name,session_date,part_of_day,client_id,type_pay,pay_at,cinema_id,cinema_room_id,count_seats,is_vip,is_3d,discount_perc,discount_amount,ticket_price_final,month,cinema_room_type,session_key,client_purchase_number,session_tickets_sold,session_revenue,occupancy_rate
0,1,190,Побег из Шоушенка,2024-07-02,вечерний,1218935,Касса,2024-07-02 17:15:49,AC1,2,28,False,False,0,0.0,190.0,7,Standart,2024-07-02 вечерний AC1 2,1,4,760.0,14.29
1,1,190,Побег из Шоушенка,2024-07-02,вечерний,1229875,Сайт,2024-07-02 19:41:49,AC1,2,28,False,False,0,0.0,190.0,7,Standart,2024-07-02 вечерний AC1 2,1,4,760.0,14.29


In [11]:
sql_bar = '''
SELECT *
FROM alpha_cinema.bar_payments
'''

In [12]:
df_bar = client.query_df(sql_bar)
df_bar.head(2)

,client_id,pay_at,name,price
0,751,2024-05-04 19:20:10,Спрайт,79
1,751,2024-05-04 19:20:10,Попкорн сладкий,290


In [5]:
# Разведывательный анализ данных (структура/качество данных)
display('Таблица tickets')
display(df_tickets.head())                                       # первые строки таблицы
df_tickets.info()                                                # структура данных и типы столбцов
display(df_tickets.isna().sum())                                 # количество пропусков
display(df_tickets.duplicated().sum())                           # количество дубликатов
display(df_tickets.describe())                                   # статистика числовых признаков (позволяет быстро проверить, нет ли "странных" записей, выбросов и тп)

display('Таблица bar')
display(df_bar.head())
df_bar.info()
display(df_bar.isna().sum())
display(df_bar.duplicated().sum())
display(df_bar.describe())

'Таблица tickets'

,film_id,price_ticket,film_name,session_date,part_of_day,client_id,type_pay,pay_at,cinema_id,cinema_room_id,count_seats,is_vip,is_3d,discount_perc,discount_amount,ticket_price_final,month,cinema_room_type,session_key,client_purchase_number,session_tickets_sold,session_revenue,occupancy_rate
0,1,190,Побег из Шоушенка,2024-07-02,вечерний,1218935,Касса,2024-07-02 17:15:49,AC1,2,28,False,False,0,0.0,190.0,7,Standart,2024-07-02 вечерний AC1 2,1,4,760.0,14.29
1,1,190,Побег из Шоушенка,2024-07-02,вечерний,1229875,Сайт,2024-07-02 19:41:49,AC1,2,28,False,False,0,0.0,190.0,7,Standart,2024-07-02 вечерний AC1 2,1,4,760.0,14.29
2,1,190,Побег из Шоушенка,2024-07-02,вечерний,1307750,Сайт,2024-07-02 14:35:52,AC1,2,28,False,False,0,0.0,190.0,7,Standart,2024-07-02 вечерний AC1 2,1,4,760.0,14.29
3,1,190,Побег из Шоушенка,2024-07-02,вечерний,1078387,Сайт,2024-07-02 03:08:58,AC1,2,28,False,False,0,0.0,190.0,7,Standart,2024-07-02 вечерний AC1 2,1,4,760.0,14.29
4,1,190,Побег из Шоушенка,2024-07-02,вечерний,1127002,Сайт,2024-07-02 10:41:30,AC2,3,20,False,False,0,0.0,190.0,7,Standart,2024-07-02 вечерний AC2 3,1,10,1852.5,50.00


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9643 entries, 0 to 9642
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   film_id                 9643 non-null   Int64         
 1   price_ticket            9643 non-null   Int64         
 2   film_name               9643 non-null   string        
 3   session_date            9643 non-null   datetime64[ns]
 4   part_of_day             9643 non-null   string        
 5   client_id               9643 non-null   Int64         
 6   type_pay                9643 non-null   string        
 7   pay_at                  9643 non-null   datetime64[ns]
 8   cinema_id               9643 non-null   string        
 9   cinema_room_id          9643 non-null   int64         
 10  count_seats             9643 non-null   Int64         
 11  is_vip                  9643 non-null   bool          
 12  is_3d                   9643 non-null   bool    

film_id                   0
price_ticket              0
film_name                 0
session_date              0
part_of_day               0
client_id                 0
type_pay                  0
pay_at                    0
cinema_id                 0
cinema_room_id            0
count_seats               0
is_vip                    0
is_3d                     0
discount_perc             0
discount_amount           0
ticket_price_final        0
month                     0
cinema_room_type          0
session_key               0
client_purchase_number    0
session_tickets_sold      0
session_revenue           0
occupancy_rate            0
dtype: int64

np.int64(0)

,film_id,price_ticket,session_date,client_id,pay_at,cinema_room_id,count_seats,discount_perc,discount_amount,ticket_price_final,month,client_purchase_number,session_tickets_sold,session_revenue,occupancy_rate
count,9643.0,9643.0,9643,9643.0,9643,9643.000000,9643.0,9643.0,9643.000000,9643.000000,9643.0,9643.000000,9643.000000,9643.000000,9643.000000
mean,29.77424,303.817277,2024-05-03 15:43:01.561754624,1604691.63362,2024-05-03 06:59:14.907601408,2.011615,23.810432,1.832936,5.566214,298.251063,4.585399,1.007363,10.732656,3259.624235,45.733835
min,1.0,190.0,2024-03-01 00:00:00,751.0,2024-02-13 19:51:08,1.000000,12.0,0.0,0.000000,161.500000,2.0,1.000000,1.000000,161.500000,2.780000
25%,20.0,190.0,2024-03-31 00:00:00,873934.5,2024-03-30 11:11:30.500000,1.000000,20.0,0.0,0.000000,190.000000,3.0,1.000000,4.000000,1160.000000,20.000000
50%,34.0,290.0,2024-05-05 00:00:00,1257971.0,2024-05-04 18:39:08,2.000000,20.0,0.0,0.000000,290.000000,5.0,1.000000,9.000000,2413.000000,41.670000
75%,37.0,340.0,2024-06-04 00:00:00,2743541.0,2024-06-03 17:34:57,3.000000,28.0,0.0,0.000000,340.000000,6.0,1.000000,15.000000,4524.000000,71.430000
max,53.0,450.0,2024-07-31 00:00:00,3096911.0,2024-07-31 19:28:18,3.000000,36.0,15.0,67.500000,450.000000,7.0,3.000000,36.000000,12915.000000,100.000000
std,13.22877,91.993517,NaN,949501.650858,NaN,0.749244,7.791234,3.825127,12.287016,91.077711,1.25194,0.086700,7.916168,2754.527498,28.669205


'Таблица bar'

NameError: name 'df_bar' is not defined

In [17]:
duplicates = df_bar[df_bar.duplicated(keep=False)] \
    .sort_values(['client_id', 'pay_at'])                          # предыдущий шаг выявил 1934 дубликатов в df_bar (11,46%)
duplicates.head(20)                                                # проверяем, что это за дубли, можно ли/нужно ли ух удалить

,client_id,pay_at,name,price
4,22166,2024-03-18 15:20:57,Красное вино,390
5,22166,2024-03-18 15:20:57,Красное вино,390
10003,27528,2024-05-18 19:33:34,Попкорн соленый,290
10004,27528,2024-05-18 19:33:34,Попкорн соленый,290
9,33974,2024-03-01 15:36:54,Попкорн соленый,290
10,33974,2024-03-01 15:36:54,Попкорн соленый,290
17,40366,2024-04-23 15:53:59,Пиво,249
18,40366,2024-04-23 15:53:59,Пиво,249
10008,42635,2024-07-10 19:53:54,Начос,359
10011,42635,2024-07-10 19:53:54,Начос,359


In [18]:
print('Дубли до очистки:', df_bar.duplicated().sum())               # выводим дубли до очистки (для сравнения)
df_bar = df_bar.drop_duplicates()                                   # удаляем полные дубли
print('Дубли после очистки:', df_bar.duplicated().sum())            # проверяем результат после очистки
print('Размер таблицы после очистки:', df_bar.shape)

Дубли до очистки: 1934
Дубли после очистки: 0
Размер таблицы после очистки: (14941, 4)


In [20]:
# Создаем признаки для аналитики покупок, загрузки сеансов и канала продаж
df_tickets['is_repeat_purchase'] = np.where(
    df_tickets['client_purchase_number'] > 1, 1, 0
)
df_tickets['is_high_occupancy'] = np.where(
    df_tickets['occupancy_rate'] >= 70, 1, 0
)
df_tickets['sales_channel'] = np.where(
    df_tickets['type_pay'] == 'Сайт','Online','Offline'
)

In [21]:
# Формируем витрину сеансов: одна строка = один сеанс
session_metrics = (
    df_tickets.groupby(['session_key'], as_index=False                                    # группируем данные по сеансам
)
    .agg(
        session_date=('session_date', 'first'),                                           # до группировки данные в развернутом виде 1 строка = 1 билет, поэтому указываем, 
        month=('month', 'first'),                                                         # какое значение берем (тк значения одинаковые, берем просто первое)
        cinema_id=('cinema_id', 'first'),
        cinema_room_id=('cinema_room_id', 'first'),
        cinema_room_type=('cinema_room_type', 'first'),
        film_id=('film_id', 'first'),
        film_name=('film_name', 'first'),
        count_seats=('count_seats', 'first'),
        session_tickets_sold=('session_tickets_sold', 'first'),
        session_revenue=('session_revenue', 'first'),
        occupancy_rate_session=('occupancy_rate', 'first'),
        avg_ticket_price=('ticket_price_final', lambda x: round(x.mean(), 2)),
        total_discount=('discount_amount', 'sum'),
        online_tickets=('sales_channel', lambda x: (x == 'Online').sum())
    )
)
session_metrics['online_share_session'] = round(                                          # доля онлайн-продаж на сеанс
    session_metrics['online_tickets'] /
    session_metrics['session_tickets_sold'] * 100,
    2
)
session_metrics.sample(5)

,session_key,session_date,month,cinema_id,cinema_room_id,cinema_room_type,film_id,film_name,count_seats,session_tickets_sold,session_revenue,occupancy_rate_session,avg_ticket_price,total_discount,online_tickets,online_share_session
285,2024-03-20 дневной AC1 2,2024-03-20,3,AC1,2,Standart,47,Касабланка,28,8,3510.0,28.57,438.75,90.0,5,62.50
1046,2024-05-12 вечерний AC1 2,2024-05-12,5,AC1,2,Standart,26,Город Бога,28,15,2793.0,53.57,186.20,57.0,6,40.00
16,2024-03-02 дневной AC1 1,2024-03-02,3,AC1,1,3D,44,Могила светлячков,20,7,3150.0,35.00,450.00,0.0,5,71.43
1770,2024-07-04 дневной AC1 2,2024-07-04,7,AC1,2,Standart,3,Тёмный рыцарь,28,2,680.0,7.14,340.00,0.0,1,50.00
1800,2024-07-08 вечерний AC3 1,2024-07-08,7,AC3,1,VIP,5,12 разгневанных мужчин,12,1,290.0,8.33,290.00,0.0,1,100.00


In [22]:
# Собираем клиентские метрики по билетам
client_metrics = (
    df_tickets.groupby('client_id', as_index=False)                                       # группируем записи по клиенту
    .agg(
        tickets_bought=('client_id', 'count'),
        sessions_visited=('session_key', 'nunique'),                                      # тк клиент мог купить 2 билета на один сеанс
        total_ticket_revenue=('ticket_price_final', 'sum'),
        avg_ticket_price=('ticket_price_final', 'mean'),
        total_discount=('discount_amount', 'sum'),
        first_purchase=('pay_at', 'min'),
        last_purchase=('pay_at', 'max')
    )
)
client_metrics.sample(5)

,client_id,tickets_bought,sessions_visited,total_ticket_revenue,avg_ticket_price,total_discount,first_purchase,last_purchase
8944,3033331,1,1,450.0,450.0,0.0,2024-05-16 22:56:15,2024-05-16 22:56:15
5271,1296880,1,1,450.0,450.0,0.0,2024-06-13 19:10:53,2024-06-13 19:10:53
1788,766189,1,1,340.0,340.0,0.0,2024-05-19 19:49:34,2024-05-19 19:49:34
9446,3076029,1,1,450.0,450.0,0.0,2024-04-15 09:48:33,2024-04-15 09:48:33
15,37940,1,1,190.0,190.0,0.0,2024-03-12 05:16:43,2024-03-12 05:16:43


In [23]:
# Сегментируем клиентов по числу посещенных сеансов
conditions = [
    client_metrics['sessions_visited'] == 1,
    client_metrics['sessions_visited'].between(2, 3),
    client_metrics['sessions_visited'] >= 4
]
choices = ['Новый', 'Повторный', 'Постоянный']
client_metrics['client_type'] = np.select(
    conditions,
    choices,
    default='Новый'
)

In [24]:
# Собираем клиентские метрики по бару
bar_client_metrics = (
    df_bar.groupby('client_id', as_index=False)
    .agg(
        bar_orders_count=('price', 'count'),
        bar_revenue=('price', 'sum'),
        bar_avg_check=('price', lambda x: x.mean().round(2))
    )
)
bar_client_metrics.sample(5)

,client_id,bar_orders_count,bar_revenue,bar_avg_check
610,565242,2,369,184.50
6002,3032301,3,1370,456.67
3648,1318961,2,328,164.00
3460,1290015,1,249,249.00
1506,867309,2,328,164.00


In [25]:
# Объединяем билетные и барные метрики по клиенту
client_metrics = client_metrics.merge(
    bar_client_metrics,
    on='client_id',
    how='left'
)
client_metrics['bar_orders_count'] = client_metrics['bar_orders_count'].fillna(0)
client_metrics['bar_revenue'] = client_metrics['bar_revenue'].fillna(0)
client_metrics['bar_avg_check'] = client_metrics['bar_avg_check'].fillna(0)
client_metrics['has_bar_purchase'] = np.where(
    client_metrics['bar_orders_count'] > 0, 1, 0
)
client_metrics['total_revenue_with_bar'] = (
    client_metrics['total_ticket_revenue'] + client_metrics['bar_revenue']
)
client_metrics.sample(5)

,client_id,tickets_bought,sessions_visited,total_ticket_revenue,avg_ticket_price,total_discount,first_purchase,last_purchase,client_type,bar_orders_count,bar_revenue,bar_avg_check,has_bar_purchase,total_revenue_with_bar
9286,3059536,1,1,190.0,190.0,0.0,2024-03-04 07:48:35,2024-03-04 07:48:35,Новый,0,0,0.00,0,190.0
3501,1029711,1,1,190.0,190.0,0.0,2024-06-26 08:10:06,2024-06-26 08:10:06,Новый,0,0,0.00,0,190.0
6078,1428930,1,1,190.0,190.0,0.0,2024-03-20 17:22:59,2024-03-20 17:22:59,Новый,3,718,239.33,1,908.0
5472,1319192,1,1,180.5,180.5,9.5,2024-06-15 16:51:14,2024-06-15 16:51:14,Новый,2,1039,519.50,1,1219.5
8981,3035432,1,1,275.5,275.5,14.5,2024-05-01 05:47:07,2024-05-01 05:47:07,Новый,3,418,139.33,1,693.5


In [27]:
# Сохраняем итоговые датасеты для загрузки в DataLens
session_metrics.to_csv('session_metrics.csv', index=False)
client_metrics.to_csv('client_metrics.csv', index=False)